# ARC-v0.30 — Post-Primary Reviewer Hardening Audit

**Purpose.** This audit addresses two reviewer-facing questions without changing any frozen v0.28/v0.29 primary:

1. **Slope vs. level:** Does the long-horizon negative representation-minus-search `H3abs` contrast imply a crossover in the *absolute utility-gap level*, or only a reversal in temporal trend ordering?
2. **Structural theory audit:** Using the already-generated ARC-v0.23 common-state replay, what empirical common-map propagation ratios are observed under the frozen **anchored** FEVER-E5 operator?

**Classification.** Post-primary reviewer-oriented analysis. It is not a new confirmatory primary and must not be used to redefine v0.28/v0.29 success criteria.

**Non-negotiable interpretation rule.** If H3 trend ordering reverses but gap-level ordering does not, report exactly that. If empirical gain estimates are null, >1, unstable, or mechanism-invariant, retain them unchanged.

In [ ]:
from pathlib import Path
from collections import defaultdict
import json, math, os, hashlib
import numpy as np
import pandas as pd

try:
    import pyarrow  # noqa
except Exception:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyarrow"])

try:
    from google.colab import drive
except ImportError:
    drive = None

SEED = 20260826
BOOTSTRAP_REPS = 10_000
rng = np.random.default_rng(SEED)

DRIVE_ROOT = Path("/content/drive/MyDrive")
if not DRIVE_ROOT.is_dir():
    if drive is None:
        raise RuntimeError("Google Drive is not mounted.")
    drive.mount("/content/drive")

ARC = DRIVE_ROOT / "rag-pq-checkpoints" / "arc-v0"

V028 = ARC / "nq-gte-operator-horizon-generalization-v028" / "20260825-181541"
V029 = ARC / "fever-e5-operator-horizon-replication-v029" / "20260826-120722"
V023 = ARC / "eq6-common-state-operator-replay-v023" / "20260822-162028"

P028 = V028 / "v028_h50_agentic_bridge_endpoints.parquet"
P029 = V029 / "v029_h50_endpoints.parquet"
P023 = V023 / "v023_full_common_state_replay.parquet"

for p in [P028, P029, P023]:
    assert p.is_file(), f"Missing: {p}"

OUT = ARC / "reviewer-hardening-v030" / "20260826-post-primary"
OUT.mkdir(parents=True, exist_ok=True)

print("v0.28:", P028)
print("v0.29:", P029)
print("v0.23:", P023)
print("OUT:", OUT)

In [ ]:
# Load only the existing analysis artifacts. No retrieval is run here.
e28 = pd.read_parquet(P028)
e29 = pd.read_parquet(P029)
r23 = pd.read_parquet(P023)

for name, df in [("v028",e28),("v029",e29),("v023",r23)]:
    print("\n", name, df.shape)
    print(df.columns.tolist())
    display(df.head(2))

In [ ]:
def pick_col(df, candidates, required=True):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f"None of {candidates} found. Available: {df.columns.tolist()}")
    return None

def paired_bootstrap(x, seed, reps=BOOTSTRAP_REPS):
    x = np.asarray(x, dtype=np.float64)
    x = x[np.isfinite(x)]
    n = len(x)
    if n == 0:
        raise ValueError("No finite observations.")
    rg = np.random.default_rng(seed)
    boots = np.empty(reps, dtype=np.float64)
    chunk = 250
    pos = 0
    while pos < reps:
        b = min(chunk, reps-pos)
        idx = rg.integers(0, n, size=(b,n), dtype=np.int32)
        boots[pos:pos+b] = x[idx].mean(axis=1)
        pos += b
    return {
        "mean": float(x.mean()),
        "ci95_low": float(np.quantile(boots, .025)),
        "ci95_high": float(np.quantile(boots, .975)),
        "n_queries": int(n),
    }

def bootstrap_mean(x, seed, reps=BOOTSTRAP_REPS):
    return paired_bootstrap(x, seed, reps)

def endpoint_schema(df):
    """
    Normalize the slightly different v0.28/v0.29 endpoint column names.

    v0.28 stores metric-prefixed fields such as:
      ndcg10_final_abs_gap
      ndcg10_final_minus_initial_abs_gap
      ndcg10_H3_abs_slope

    v0.29 stores shorter fields such as:
      abs_gap_at_H
      abs_gap_final_minus_initial
      H3_abs_slope
    """
    return {
        "qid": pick_col(df, ["query_id","qid"]),
        "mech": pick_col(df, ["mechanism"]),
        "op": pick_col(df, ["operator"]),
        "H": pick_col(df, ["H","horizon"]),
        "family": pick_col(df, ["family"], required=False),
        "policy": pick_col(df, ["config_key","policy_key","config"], required=False),

        # gap level at the audited horizon
        "gapH": pick_col(df, [
            "abs_gap_at_H",
            "ndcg_abs_gap_at_H",
            "abs_gap_H",
            "ndcg10_final_abs_gap",
            "ndcg10_abs_gap_at_H",
        ]),

        # final-minus-initial absolute nDCG gap
        "R1": pick_col(df, [
            "abs_gap_final_minus_initial",
            "ndcg_abs_gap_final_minus_initial",
            "R1_abs_gap",
            "ndcg10_final_minus_initial_abs_gap",
        ], required=False),

        # H3 absolute-gap slope
        "H3": pick_col(df, [
            "H3_abs_slope",
            "H3abs",
            "H3_abs",
            "ndcg10_H3_abs_slope",
        ], required=False),
    }

print("v028 schema:", endpoint_schema(e28))
print("v029 schema:", endpoint_schema(e29))

## A. Cross-dataset slope-vs-level audit

For each dataset, operator, horizon, and mechanism, first average the *gap level* within query over the frozen long-horizon policy subset. The independent unit remains the query. We then compute the paired query-level contrast:

\[
D_H = G_{\mathrm{repr}}(H)-G_{\mathrm{search}}(H).
\]

A negative H3 contrast and positive \(D_H\) are not contradictory: they mean representation-induced separation is relaxing faster, while its absolute gap level remains larger.

In [ ]:
def make_level_audit(df, dataset, seed_offset):
    s = endpoint_schema(df)
    rows=[]
    for op in sorted(df[s["op"]].astype(str).unique()):
        for H in sorted(pd.to_numeric(df[s["H"]]).unique()):
            sub = df[(df[s["op"]].astype(str)==str(op)) & (pd.to_numeric(df[s["H"]])==H)]
            qm={}
            for mech in ["representation","search_effort"]:
                x = sub[sub[s["mech"]].astype(str)==mech]
                qm[mech] = x.groupby(s["qid"])[s["gapH"]].mean().sort_index()
            common = qm["representation"].index.intersection(qm["search_effort"].index)
            if len(common)==0:
                continue
            rep = qm["representation"].loc[common].to_numpy(float)
            sea = qm["search_effort"].loc[common].to_numpy(float)
            diff = rep-sea
            sr = bootstrap_mean(rep, SEED+seed_offset+int(H)*10+1)
            ss = bootstrap_mean(sea, SEED+seed_offset+int(H)*10+2)
            sd = paired_bootstrap(diff, SEED+seed_offset+int(H)*10+3)
            rows.append({
                "dataset":dataset,"operator":op,"H":int(H),
                "representation_gap_mean":sr["mean"],
                "representation_ci_low":sr["ci95_low"],"representation_ci_high":sr["ci95_high"],
                "search_gap_mean":ss["mean"],
                "search_ci_low":ss["ci95_low"],"search_ci_high":ss["ci95_high"],
                "rep_minus_search_gap":sd["mean"],
                "diff_ci_low":sd["ci95_low"],"diff_ci_high":sd["ci95_high"],
                "n_queries":sd["n_queries"],
            })
    return pd.DataFrame(rows)

level = pd.concat([
    make_level_audit(e28, "NQ-GTE", 280000),
    make_level_audit(e29, "FEVER-E5", 290000),
], ignore_index=True)

level.to_csv(OUT/"v030_cross_dataset_gap_level_audit.csv", index=False)
display(level[level["operator"].astype(str).eq("recursive")].sort_values(["dataset","H"]))

# Explicitly test whether the paired level contrast crosses zero.
classification=[]
for (ds,op),g in level.groupby(["dataset","operator"]):
    signs=[]
    for _,r in g.sort_values("H").iterrows():
        if r.diff_ci_low>0: signs.append("positive")
        elif r.diff_ci_high<0: signs.append("negative")
        else: signs.append("unresolved")
    classification.append({
        "dataset":ds,"operator":op,
        "horizons":",".join(map(str,g.sort_values("H").H.astype(int))),
        "level_signs":",".join(signs),
        "gap_level_crossover_detected": ("positive" in signs and "negative" in signs),
    })
class_df=pd.DataFrame(classification)
class_df.to_csv(OUT/"v030_gap_level_crossover_classification.csv", index=False)
display(class_df)

In [ ]:
# R1 = final-minus-initial gap. If persisted directly, use it.
def make_R1_audit(df, dataset, seed_offset):
    s=endpoint_schema(df)
    if s["R1"] is None:
        print(dataset, "has no persisted R1 column; skipping.")
        return pd.DataFrame()
    rows=[]
    for op in sorted(df[s["op"]].astype(str).unique()):
        for H in sorted(pd.to_numeric(df[s["H"]]).unique()):
            sub=df[(df[s["op"]].astype(str)==str(op)) & (pd.to_numeric(df[s["H"]])==H)]
            qm={}
            for mech in ["representation","search_effort"]:
                qm[mech]=sub[sub[s["mech"]].astype(str)==mech].groupby(s["qid"])[s["R1"]].mean().sort_index()
            common=qm["representation"].index.intersection(qm["search_effort"].index)
            if len(common)==0: continue
            diff=(qm["representation"].loc[common]-qm["search_effort"].loc[common]).to_numpy(float)
            st=paired_bootstrap(diff, SEED+seed_offset+int(H))
            rows.append({"dataset":dataset,"operator":op,"H":int(H),"estimand":"R1_rep_minus_search",**st})
    return pd.DataFrame(rows)

r1 = pd.concat([
    make_R1_audit(e28,"NQ-GTE",281000),
    make_R1_audit(e29,"FEVER-E5",291000),
], ignore_index=True)
r1.to_csv(OUT/"v030_cross_dataset_R1_audit.csv", index=False)
display(r1[(r1.operator.astype(str)=="recursive")].sort_values(["dataset","H"]))

## B. Empirical common-map propagation audit from ARC-v0.23

ARC-v0.23 stores the common-state decomposition

\[
v_{\text{prop}} = T_H(q_H^t)-T_H(q_L^t), \qquad
v_{\text{direct}} = T_H(q_L^t)-T_L(q_L^t).
\]

For rounds \(t\ge1\), the prior round's stored total state separation is the input separation to the next common-map replay. We therefore compute the post-primary empirical ratio

\[
\widehat{\rho}^{\,map}_{q,\pi,t}
=
\frac{\|v_{\text{prop}}(t)\|}
{\|v_{\text{total}}(t-1)\|}.
\]

**Scope:** v0.23 uses the frozen FEVER-E5 **anchored** operator. This audit empirically probes common-map propagation in that operator only; it does **not** claim to estimate the recursive memory term or a global Lipschitz constant.

In [ ]:
# Identify v0.23 schema robustly.
qid = pick_col(r23, ["query_id","qid"])
mech = pick_col(r23, ["mechanism"])
rnd = pick_col(r23, ["round","t"])
prop = pick_col(r23, ["state_prop_norm"])
total = pick_col(r23, ["state_total_norm"])
direct = pick_col(r23, ["state_direct_norm"])

candidate_group_cols = ["config_key","policy_key","family","alpha","k","temperature"]
group_cols = [qid, mech] + [c for c in candidate_group_cols if c in r23.columns]
print("group keys:", group_cols)

x=r23.copy()
x[rnd]=pd.to_numeric(x[rnd])
x=x.sort_values(group_cols+[rnd]).reset_index(drop=True)
x["prev_state_total_norm"] = x.groupby(group_cols, dropna=False)[total].shift(1)

# Round 0 has no prior state separation; do not invent a ratio.
g = x[(x[rnd] >= 1) & np.isfinite(x["prev_state_total_norm"])].copy()

DENOM_FLOOR = 1e-6
g["denom_ok"] = g["prev_state_total_norm"] >= DENOM_FLOOR
retention = g.groupby(mech)["denom_ok"].mean()
print("retained fraction by mechanism:")
print(retention)

g = g[g["denom_ok"]].copy()
g["rho_map"] = g[prop] / g["prev_state_total_norm"]
assert np.isfinite(g["rho_map"]).all()

# Aggregate policy realizations within query before inference.
per_query_round = (
    g.groupby([qid,mech,rnd], as_index=False)
     .agg(
         rho_map=("rho_map","mean"),
         prop_norm=(prop,"mean"),
         prev_total=("prev_state_total_norm","mean"),
         direct_norm=(direct,"mean"),
         n_policy_rounds=("rho_map","size"),
     )
)

# Round-specific query-level summaries.
rows=[]
for (m,t),z in per_query_round.groupby([mech,rnd]):
    vals=z["rho_map"].to_numpy(float)
    st=bootstrap_mean(vals, SEED+300000+int(t)*100+{"e5_representation":1,"e5_nprobe":2,"e5_hnsw":3}.get(str(m),9))
    rows.append({
        "mechanism":m,"round":int(t),
        "mean_query_rho":st["mean"],"ci95_low":st["ci95_low"],"ci95_high":st["ci95_high"],
        "median_query_rho":float(np.median(vals)),
        "q25":float(np.quantile(vals,.25)),"q75":float(np.quantile(vals,.75)),
        "fraction_query_rho_gt_1":float(np.mean(vals>1)),
        "n_queries":len(vals),
    })
gain_round=pd.DataFrame(rows)
gain_round.to_csv(OUT/"v030_empirical_common_map_gain_by_round.csv", index=False)
display(gain_round.sort_values(["mechanism","round"]))

# Overall across rounds 1..3: first average within query, then bootstrap queries.
pq=per_query_round.groupby([qid,mech],as_index=False)["rho_map"].mean()
rows=[]
for m,z in pq.groupby(mech):
    vals=z.rho_map.to_numpy(float)
    st=bootstrap_mean(vals, SEED+310000+{"e5_representation":1,"e5_nprobe":2,"e5_hnsw":3}.get(str(m),9))
    rows.append({
        "mechanism":m,**st,
        "median_query_rho":float(np.median(vals)),
        "q25":float(np.quantile(vals,.25)),"q75":float(np.quantile(vals,.75)),
        "fraction_query_rho_gt_1":float(np.mean(vals>1)),
    })
gain_overall=pd.DataFrame(rows)
gain_overall.to_csv(OUT/"v030_empirical_common_map_gain_overall.csv", index=False)
display(gain_overall)

In [ ]:
# Paired mechanism contrasts on the query-averaged empirical gain.
wide=pq.pivot(index=qid,columns=mech,values="rho_map")
comparisons=[
    ("e5_representation","e5_nprobe"),
    ("e5_representation","e5_hnsw"),
    ("e5_nprobe","e5_hnsw"),
]
rows=[]
for a,b in comparisons:
    if a not in wide or b not in wide: continue
    z=wide[[a,b]].dropna()
    d=(z[a]-z[b]).to_numpy(float)
    st=paired_bootstrap(d, SEED+320000+len(rows))
    rows.append({"contrast":f"{a}_minus_{b}",**st})
gain_diff=pd.DataFrame(rows)
gain_diff.to_csv(OUT/"v030_empirical_common_map_gain_pairwise.csv", index=False)
display(gain_diff)

In [ ]:
# Denominator-floor sensitivity: the ratio should not be an artifact of nearly-zero prior separation.
sens=[]
for floor in [0.0, 1e-6, 1e-4, 1e-3, 1e-2]:
    y=x[(x[rnd]>=1) & np.isfinite(x["prev_state_total_norm"]) & (x["prev_state_total_norm"]>floor)].copy()
    y["rho_map"]=y[prop]/y["prev_state_total_norm"]
    qr=y.groupby([qid,mech],as_index=False)["rho_map"].mean()
    for m,z in qr.groupby(mech):
        sens.append({
            "denom_floor":floor,"mechanism":m,
            "n_queries":z[qid].nunique(),
            "mean_query_rho":float(z.rho_map.mean()),
            "median_query_rho":float(z.rho_map.median()),
            "fraction_query_rho_gt_1":float((z.rho_map>1).mean()),
        })
sens=pd.DataFrame(sens)
sens.to_csv(OUT/"v030_empirical_gain_denominator_sensitivity.csv", index=False)
display(sens)

In [ ]:
# Consolidated paper-facing classification. This does not alter any frozen primary.
rec = level[level.operator.astype(str).eq("recursive")].copy()
h50 = rec[rec.H.eq(50)].set_index("dataset")
r1h50 = r1[(r1.operator.astype(str)=="recursive") & (r1.H==50)].set_index("dataset") if len(r1) else pd.DataFrame()

report = {
    "study_id":"ARC-v0.30",
    "classification":"post-primary reviewer hardening audit",
    "frozen_primaries_unchanged":True,
    "gap_level_h50":{
        ds:{
            "rep_minus_search":float(row.rep_minus_search_gap),
            "ci95":[float(row.diff_ci_low),float(row.diff_ci_high)],
            "level_crossover_at_h50": bool(row.diff_ci_high < 0),
        } for ds,row in h50.iterrows()
    },
    "interpretation_rule":(
        "A negative H3 representation-minus-search contrast is a temporal trend-order reversal. "
        "It is called a gap-level crossover only if the paired gap-level contrast itself becomes negative."
    ),
    "v023_gain_scope":"FEVER-E5 anchored common-state replay only; not a recursive/global Lipschitz estimate.",
}
if len(r1):
    report["R1_h50"]={
        ds:{"mean":float(row["mean"]),"ci95":[float(row.ci95_low),float(row.ci95_high)]}
        for ds,row in r1h50.iterrows()
    }

(OUT/"v030_reviewer_hardening_report.json").write_text(json.dumps(report,indent=2,sort_keys=True))
print(json.dumps(report,indent=2))
print("\nOutputs:")
for p in sorted(OUT.glob("v030_*")):
    print(p.name, p.stat().st_size)

## Paper-facing interpretation guardrails

After execution, use the following wording rules:

- If the H3 contrast changes sign but `rep_minus_search_gap` remains positive, call it **H3 trend-order reversal under differential relaxation**, not a gap-level reversal.
- If the gap-level contrast itself becomes significantly negative, explicitly distinguish the horizon at which that *level crossover* appears.
- The v0.23 gain audit is evidence about **anchored common-map state propagation**, not a global theorem and not a direct predictor of H3 utility sign.
- Equation (7) remains a state-space structural bound. It does not by itself predict utility-gap trend sign, gap-level crossover, or the transition horizon.